# AXIS QUANT — 工业级双均线动量突破策略实战
## 运行环境: Google Colab (Python 3.10+ / GPU/TPU 加速)
本笔记本为 [AXIS QUANT 量化交易平台](https://xinkuangsi-spec.github.io/axis-quant/) 官方配套实战代码。

### 学习目标:
1. 通过 `yfinance` 获取标普 500 (SPY) 与比特币 (BTC-USD) 历史行情数据；
2. 构建快速 SMA 与慢速 SMA 指标并捕捉黄金交叉/死亡交叉；
3. 计算策略累计收益率、年化夏普比率 (Sharpe Ratio) 与最大回撤 (Max Drawdown)；
4. 绘制权益资产曲线与买卖信号图表。

In [ ]:
# 1. 安装核心量化依赖库
!pip install -q yfinance pandas numpy matplotlib

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# 2. 获取数据
ticker = 'SPY'
df = yf.download(ticker, start='2021-01-01', end='2026-01-01')
df = df[['Close']].copy()
df.dropna(inplace=True)
print(f'成功下载 {ticker} 数据，共 {len(df)} 根 K 线。')
df.head()

In [ ]:
# 3. 构造双均线策略信号
fast_period = 8
slow_period = 24

df['SMA_Fast'] = df['Close'].rolling(window=fast_period).mean()
df['SMA_Slow'] = df['Close'].rolling(window=slow_period).mean()

# 信号: 1 持仓做多, 0 空仓观望
df['Signal'] = np.where(df['SMA_Fast'] > df['SMA_Slow'], 1.0, 0.0)
# 产生持仓变动的交易日
df['Position'] = df['Signal'].diff()

# 4. 收益率与绩效计算
df['Market_Return'] = df['Close'].pct_change()
df['Strategy_Return'] = df['Signal'].shift(1) * df['Market_Return']

# 累计净值
df['Cumulative_Market'] = (1 + df['Market_Return']).cumprod()
df['Cumulative_Strategy'] = (1 + df['Strategy_Return']).cumprod()

# 年化夏普比率计算 (无风险利率设定 2%)
rf_daily = 0.02 / 252
excess_ret = df['Strategy_Return'].dropna() - rf_daily
sharpe_ratio = np.sqrt(252) * (excess_ret.mean() / excess_ret.std())

# 最大回撤计算
running_max = np.maximum.accumulate(df['Cumulative_Strategy'])
drawdown = (running_max - df['Cumulative_Strategy']) / running_max
max_drawdown = drawdown.max()

total_strategy_ret = (df['Cumulative_Strategy'].iloc[-1] - 1) * 100
total_market_ret = (df['Cumulative_Market'].iloc[-1] - 1) * 100

print('================= 策略回测绩效报告 =================')
print(f'策略累计总收益率: {total_strategy_ret:.2f}%')
print(f'基准买入持有收益: {total_market_ret:.2f}%')
print(f'年化夏普比率 (Sharpe): {sharpe_ratio:.2f}')
print(f'历史最大回撤率 (MDD): {max_drawdown*100:.2f}%')
print('====================================================')

In [ ]:
# 5. 绘制可视化对比图表
plt.figure(figsize=(14, 7), dpi=120)
plt.plot(df.index, df['Cumulative_Strategy'], label='AXIS SMA Strategy', color='#1F44FF', lw=2)
plt.plot(df.index, df['Cumulative_Market'], label='SPY Buy & Hold', color='#6B6A66', lw=1.5, ls='--')
plt.title(f'{ticker} 双均线动量策略与基准净值走势对比', fontsize=14, fontweight='bold')
plt.xlabel('日期')
plt.ylabel('累计净值 (起始=1.0)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()